## 1. import

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier, Pool

## 2. config

In [2]:
class CFG:
    SEED = 42
    N_SPLITS = 5
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    MODEL_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": 3000,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 5.0,
        "random_strength": 1.0,
        "bagging_temperature": 0.0,
        "grow_policy": "SymmetricTree",
        "bootstrap_type": "Bayesian",
        "random_seed": SEED,
        "verbose": 200,
        "early_stopping_rounds": 200,
        "task_type": "GPU" if torch.cuda.is_available() else "CPU",
    }

## 3. column groups

In [3]:
# =========================
# Column groups
# =========================
NUM_COLS_BASE = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

CAT_COLS_BASE = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Region",
]

BIN_COLS_BASE = [
    "Mulching_Used",
]

CAT_ALL_BASE = CAT_COLS_BASE + BIN_COLS_BASE
FEATURE_COLS_BASE = NUM_COLS_BASE + CAT_COLS_BASE + BIN_COLS_BASE

LOG_SOURCE_COLS = [
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

## 4. utility

In [4]:
# =========================
# Utilities
# =========================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def cast_dtypes(train_df, test_df):
    for col in CAT_ALL_BASE:
        train_df[col] = train_df[col].astype(str)
        test_df[col] = test_df[col].astype(str)

    for col in NUM_COLS_BASE:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

    return train_df, test_df

## 5. load data

In [5]:
seed_everything(CFG.SEED)

train_df = pd.read_csv(CFG.TRAIN_PATH)
test_df = pd.read_csv(CFG.TEST_PATH)

train_df, test_df = cast_dtypes(train_df, test_df)

print(train_df.shape, test_df.shape)

(630000, 21) (270000, 20)


## 6. label encode

In [6]:
le = LabelEncoder()
y = le.fit_transform(train_df[CFG.TARGET_COL])
class_names = list(le.classes_)
n_classes = len(class_names)

print("classes:", class_names)

classes: ['High', 'Low', 'Medium']


## 7. LOG feature 関数

In [7]:
def add_log_features(df: pd.DataFrame, source_cols):
    df = df.copy()
    created_cols = []

    for col in source_cols:
        x = pd.to_numeric(df[col], errors="coerce").astype(float)

        # 安全のため負値は NaN に落とす
        x_nonneg = x.where(x >= 0, np.nan)

        log1p_x = np.log1p(x_nonneg)
        sqrt_x = np.sqrt(x_nonneg)
        is_zero = (x_nonneg == 0).astype(float)

        # 元値との比ではなく差分にして暴れにくくする
        diff_log1p = x_nonneg - log1p_x

        new_map = {
            f"log__{col}__log1p": log1p_x,
            f"log__{col}__sqrt": sqrt_x,
            f"log__{col}__is_zero": is_zero,
            f"log__{col}__diff_log1p": diff_log1p,
        }

        for new_col, values in new_map.items():
            df[new_col] = values
            created_cols.append(new_col)

    return df, created_cols

## 8. LOG feature 作成

In [8]:
train_df, log_cols = add_log_features(train_df, LOG_SOURCE_COLS)
test_df, _ = add_log_features(test_df, LOG_SOURCE_COLS)

FEATURE_COLS_LOG = FEATURE_COLS_BASE + log_cols

print("LOG_SOURCE_COLS:", LOG_SOURCE_COLS)
print("n log cols:", len(log_cols))
print("log cols sample:", log_cols[:10])

LOG_SOURCE_COLS: ['Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
n log cols: 32
log cols sample: ['log__Soil_Moisture__log1p', 'log__Soil_Moisture__sqrt', 'log__Soil_Moisture__is_zero', 'log__Soil_Moisture__diff_log1p', 'log__Organic_Carbon__log1p', 'log__Organic_Carbon__sqrt', 'log__Organic_Carbon__is_zero', 'log__Organic_Carbon__diff_log1p', 'log__Electrical_Conductivity__log1p', 'log__Electrical_Conductivity__sqrt']


## 9. 欠損埋め

CatBoost は数値欠損を扱えますが、念のため極端な壊れ方を避けるならそのままで構いません。
ここではそのまま進めます。

## 10. CV

In [9]:
X = train_df[FEATURE_COLS_LOG].copy()
X_test = test_df[FEATURE_COLS_LOG].copy()

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

oof_proba = np.zeros((len(train_df), n_classes), dtype=np.float32)
test_proba = np.zeros((len(test_df), n_classes), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print("=" * 60)
    print(f"fold {fold}")

    X_tr = X.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    train_pool = Pool(
        data=X_tr,
        label=y_tr,
        cat_features=CAT_ALL_BASE,
    )
    valid_pool = Pool(
        data=X_va,
        label=y_va,
        cat_features=CAT_ALL_BASE,
    )
    test_pool = Pool(
        data=X_test,
        cat_features=CAT_ALL_BASE,
    )

    model = CatBoostClassifier(**CFG.MODEL_PARAMS)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
    )

    va_proba = model.predict_proba(valid_pool)
    te_proba = model.predict_proba(test_pool)

    oof_proba[va_idx] = va_proba
    test_proba += te_proba / CFG.N_SPLITS

    va_pred = np.argmax(va_proba, axis=1)
    fold_bacc = balanced_accuracy_score(y_va, va_pred)
    fold_scores.append(fold_bacc)

    print(f"fold {fold} balanced_accuracy = {fold_bacc:.6f}")

fold 1
0:	learn: 1.0374400	test: 1.0373987	best: 1.0373987 (0)	total: 120ms	remaining: 5m 59s
200:	learn: 0.0643225	test: 0.0645791	best: 0.0645791 (200)	total: 3.17s	remaining: 44.2s
400:	learn: 0.0626275	test: 0.0632921	best: 0.0632921 (400)	total: 5.86s	remaining: 38s
600:	learn: 0.0613752	test: 0.0625815	best: 0.0625815 (600)	total: 8.53s	remaining: 34.1s
800:	learn: 0.0601053	test: 0.0620224	best: 0.0620223 (799)	total: 11.2s	remaining: 30.8s
1000:	learn: 0.0590004	test: 0.0614919	best: 0.0614919 (1000)	total: 13.9s	remaining: 27.8s
1200:	learn: 0.0581283	test: 0.0611648	best: 0.0611639 (1198)	total: 16.6s	remaining: 24.9s
1400:	learn: 0.0572970	test: 0.0609013	best: 0.0608996 (1397)	total: 19.4s	remaining: 22.1s
1600:	learn: 0.0563893	test: 0.0606492	best: 0.0606492 (1600)	total: 22.1s	remaining: 19.3s
1800:	learn: 0.0556213	test: 0.0604539	best: 0.0604536 (1798)	total: 24.7s	remaining: 16.4s
2000:	learn: 0.0547537	test: 0.0602708	best: 0.0602708 (2000)	total: 27.4s	remaining: 13

## 11. summary

In [10]:
print("=" * 60)
print("fold scores:", [round(s, 6) for s in fold_scores])
print(f"cv mean balanced_accuracy = {np.mean(fold_scores):.6f}")
print(f"cv std  balanced_accuracy = {np.std(fold_scores):.6f}")

oof_pred = np.argmax(oof_proba, axis=1)
oof_bacc = balanced_accuracy_score(y, oof_pred)
print(f"OOF balanced_accuracy = {oof_bacc:.6f}")

print("\nPer-class recall on OOF:")
for cls_idx, cls_name in enumerate(class_names):
    mask = (y == cls_idx)
    recall = (oof_pred[mask] == y[mask]).mean()
    print(f"{cls_name}: {recall:.6f}")

fold scores: [np.float64(0.957658), np.float64(0.961555), np.float64(0.959753), np.float64(0.959261), np.float64(0.957761)]
cv mean balanced_accuracy = 0.959198
cv std  balanced_accuracy = 0.001436
OOF balanced_accuracy = 0.959198

Per-class recall on OOF:
High: 0.909848
Low: 0.994285
Medium: 0.973460


## 12. save

In [11]:
test_pred = np.argmax(test_proba, axis=1)
test_label = le.inverse_transform(test_pred)

sub_df = pd.DataFrame({
    CFG.ID_COL: test_df[CFG.ID_COL],
    CFG.TARGET_COL: test_label
})

np.save("oof_catboost_log_min_proba.npy", oof_proba)
np.save("pred_catboost_log_min_proba.npy", test_proba)
sub_df.to_csv("submission_catboost_log_min.csv", index=False)

print("\nSaved:")
print("- oof_catboost_log_min_proba.npy")
print("- pred_catboost_log_min_proba.npy")
print("- submission_catboost_log_min.csv")


Saved:
- oof_catboost_log_min_proba.npy
- pred_catboost_log_min_proba.npy
- submission_catboost_log_min.csv
